In [0]:
from delta.tables import *
import pyspark.sql.functions as F
from datetime import datetime

In [0]:
landing_path = "/Volumes/databricks_practice/default/landing/raw_kaggle/Nike_AT.csv"
bronze_table = "databricks_practice.bronze.bronze_table"

In [0]:
# Read Landing data from Unity Catalog
df_landing = spark.read.format("csv") \
    .option("header", "true") \
    .load(landing_path) \
    .selectExpr(
        "*",
        "_metadata.file_path as file_path",
        "_metadata.file_modification_time as file_mod_time"
    )

    

In [0]:
# Verify last file modification from bronze_table
try:
    last_ts = (
        spark.table(bronze_table)
             .agg(F.max("file_mod_time"))
             .first()[0]
    )
except:
    last_ts = None


In [0]:
# Check if the landing has new data (modification file > last time processed)
if last_ts is not None:
    df_bronze = df_landing.filter(F.col("file_mod_time") > F.lit(last_ts))
else:
    df_bronze = df_landing   # first time it will load all the data


In [0]:
# Add date and partition columns
df_bronze = df_bronze \
    .withColumn("dt_ingestion", F.current_timestamp()) \
    .withColumn("dt_ingestion_partition", F.to_date(F.col("dt_ingestion")))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS databricks_practice.bronze")

In [0]:
# Append - Bronze table

df_bronze.write.format("delta") \
    .mode("append") \
    .partitionBy("dt_ingestion_partition") \
    .saveAsTable(bronze_table)

In [0]:
%sql
SELECT * FROM bronze.bronze_table LIMIT 10

In [0]:
spark.sql("SHOW Partitions bronze.bronze_table").show()

In [0]:
# Rows Check for duplicate data
print(spark.table("bronze.bronze_table").count())